---
title: "Part 12: Presenting Data with Great Tables"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/12-great-tables.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/12-great-tables.ipynb)


You run the analysis, produce the numbers, and paste `df.head()` into the report. The stakeholder replies: "Which column is which? Why are the numbers so long? Can you highlight the failing programs?" A plain DataFrame output served you during exploration. It doesn't serve a reader in a report.

Great Tables (`great_tables`) bridges that gap: it wraps a DataFrame in a fluent API and produces publication-ready HTML tables -- precise formatting, readable labels, conditional highlighting, and summary rows -- with no CSS knowledge required. Parts 8-11 built the data; this notebook builds the presentation.

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

In [ ]:
from great_tables import GT, loc, md as gt_md, nanoplot_options, style
import numpy as np
import pandas as pd
import polars as pl

from ark.plot.gt_style import metrics_report, themed_gt
from ark.plot.tokens import PRIMARY, SUCCESS, SURFACE_MUTED

df = pd.read_csv("data/university_analytics.csv")
df["average_marks"] = (df["midterm_score"] + df["final_score"] + df["project_score"]) / 3
df.head(3)

## The Last Mile of a Data Story

You have run the analysis. You have the numbers: pass rates by school, score distributions by program, trend lines across semesters. Now your manager asks for a report : something to put in front of a stakeholder, not a developer. You open a notebook, call `df.head()`, and stare at a grey monospace grid with no hierarchy, no colour, no units, and no sense of which numbers matter.

`df.head()` serves you in a notebook. It doesn't serve anyone else.

The gap between "I have the result" and "I can communicate the result" is called the last mile of a data story. It is where a lot of analysis work quietly disappears: correct findings, buried in formatting nobody wanted to read. **Great Tables** ([posit-dev.github.io/great-tables](https://posit-dev.github.io/great-tables/)) is the Python library that closes that gap. It wraps a pandas DataFrame in a fluent API : one that mirrors R's `{gt}` package : and produces publication-ready HTML tables with column spanners, colour scales, embedded sparklines, and controlled footnotes.

### How it compares to other table tools

| Tool | Output | Strengths | When to use instead |
| --- | --- | --- | --- |
| **Great Tables** ([posit-dev.github.io/great-tables](https://posit-dev.github.io/great-tables/)) | HTML | Full layout control, colour scales, sparklines, `{gt}`-compatible API | Reports, notebooks, any HTML output |
| **pandas Styler** ([pandas docs](https://pandas.pydata.org/docs/user_guide/style.html)) | HTML | Built-in, no extra install, fast for simple highlighting | Quick colouring when GT is overkill |
| **tabulate** ([tabulate on PyPI](https://pypi.org/project/tabulate/)) | Text, Markdown, HTML, LaTeX | Lightweight, great for terminal or Markdown output | CLI output, `.md` files |
| **rich** ([rich.readthedocs.io](https://rich.readthedocs.io)) | Terminal | Beautiful terminal tables, progress bars | Terminal-only display |
| **itables** ([mwouts.github.io/itables](https://mwouts.github.io/itables/)) | Interactive HTML | Sort, filter, search in notebook | Exploratory analysis, large tables |

### Already in your environment

```bash
uv add great-tables          # for a standalone project
```

Official docs: [posit-dev.github.io/great-tables/articles/intro](https://posit-dev.github.io/great-tables/articles/intro.html)

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 12 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Explain when a table is the right choice over a chart | Sec. 1 |
| 2 | Wrap a DataFrame with `GT()` and apply the project brand with `themed_gt()` | Sec. 2 |
| 3 | Format numbers, percentages, and missing values with `fmt_*` methods | Sec. 3 |
| 4 | Add readable column labels with `cols_label` and group related columns with `tab_spanner` | Sec. 4 |
| 5 | Target cells with `loc` and apply styling with `tab_style` | Sec. 5 |
| 6 | Add grand summary rows with `grand_summary_rows` | Sec. 6 |
| 7 | Build a model comparison table with `metrics_report()` | Sec. 7 |
:::


## 1. When Tables Beat Charts

A chart compresses a distribution into shape: it shows a trend, a cluster, or an outlier at a glance. A table preserves exact values so a reader can answer a precise question: *which course has the highest midterm average?* or *by how many points does one program outperform another?*

Use a table when:
- Readers will look up a specific row or compare two exact values
- The differences between groups are small and a chart would compress them into noise
- A report or stakeholder document needs a citable number, not an impression

Use a chart when:
- You want to show a trend, a distribution, or a relationship across many data points
- The pattern matters more than the individual values


<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Tables serve lookup; charts serve pattern recognition</span><br><br>
Neither replaces the other. A data storytelling section (Part 7) shows a trend with a chart. A summary report shows the underlying numbers in a table. The combination answers both the <em>what happened</em> and the <em>by exactly how much</em>.
</div>

## 2. Your First Styled Table

Every Great Tables workflow starts with `GT(df)`: wrapping a pandas DataFrame in the Great Tables object. From there you chain methods to add structure and styling. On its own, `GT(df)` renders a minimal unstyled table. `themed_gt()` applies the project's brand: column header background, font, border colours, and alternating row stripes: one call at the end of the chain.

The first example is a summary of mean scores by gender:


In [ ]:
summary = (
    df.groupby("gender")
    .agg(
        n_students=("student_id", "count"),
        midterm=("midterm_score", "mean"),
        final=("final_score", "mean"),
        project=("project_score", "mean"),
        fail_rate=("passed", lambda x: (~x).mean()),
    )
    .reset_index()
    .round(2)
)
summary

`GT(df)` alone already renders a table, but column names are raw and values have no formatting. Wrapping it in `themed_gt()` applies the brand while `.tab_header()` adds a title and subtitle:


In [ ]:
table = (
    GT(summary)
    .tab_header(
        title=gt_md("**Mean Exam Scores by Gender**"),
        subtitle="Students with complete score records across all three components",
    )
    .tab_source_note("Source: DS-MLOps university analytics dataset · 2,400 rows")
)
themed_gt(table, n_rows=len(summary))

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: The chain always ends with <code>themed_gt()</code></span><br><br>
<code>themed_gt()</code> applies brand-wide options (<code>tab_options</code>) and text styling. Call it <em>last</em>, after all structural methods (<code>tab_header</code>, <code>cols_label</code>, <code>tab_spanner</code>, etc.) so it can apply consistently across everything you have added.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - First Styled Table</span><br><br>
<b>Goal:</b> Group <code>df</code> by <code>program</code> instead of <code>gender</code>, compute the same five aggregates, then wrap with <code>GT</code> and <code>themed_gt</code>. Add a title that identifies the program.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>program_summary = df.groupby("program").agg(...).reset_index().round(2)
GT(program_summary).tab_header(title=gt_md("**...**"), subtitle="...").pipe(themed_gt, n_rows=len(program_summary))</pre>
</div>

In [ ]:
# TODO: build program_summary and display with GT + themed_gt
...

## 3. Formatting Values and Labelling Columns

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Format numbers to signal meaningful precision, not raw float noise</span><br><br>
A pass rate of <code>0.87654</code> tells the reader nothing that <code>87.7%</code> doesn't -- and the extra digits imply a precision the data doesn't have. Great Tables formats numbers through chainable <code>fmt_*()</code> methods: <code>fmt_number(decimals=1)</code>, <code>fmt_percent(decimals=1)</code>, <code>fmt_integer()</code>. These are display-only transforms -- the underlying DataFrame is unchanged.
</div>

Raw floats in a table communicate false precision: a pass rate of `0.87654` signals noise, not information. Great Tables `fmt_*` methods format each column's values to the right precision for its type, and `cols_label` replaces machine-readable column names with reader-facing ones.

The four formatting methods used most in DS tables:
- `fmt_number(columns, decimals)`: round to `decimals` places
- `fmt_integer(columns)`: strip decimal point, add thousands separator
- `fmt_percent(columns, decimals)`: multiply by 100 and append `%`
- `fmt_missing(columns, missing_text)`: replace `NaN` with a readable label


<div style='background:#EAF7F0;border-left:5px solid #059669;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#059669;font-weight:bold'><i class="bi bi-journal-code"></i> Example: fmt_percent turns 0.913 into 91.3%</span><br><br>
Without formatting, <code>fail_rate=0.04</code> reads as a raw proportion. With <code>fmt_percent(columns='fail_rate', decimals=1)</code>, the same cell displays as <code>4.0%</code>: the reader doesn't need to mentally multiply by 100.
</div>

In [ ]:
formatted = (
    GT(summary)
    .tab_header(
        title=gt_md("**Mean Exam Scores by Gender**"),
        subtitle="All figures rounded to one decimal place",
    )
    .cols_label(
        gender="Gender",
        n_students="Students",
        midterm="Midterm",
        final="Final",
        project="Project",
        fail_rate="Fail Rate",  # noqa: S106
    )
    .fmt_integer(columns="n_students")
    .fmt_number(columns=["midterm", "final", "project"], decimals=1)
    .fmt_percent(columns="fail_rate", decimals=1)  # noqa: S106
    .tab_source_note("Source: DS-MLOps university analytics dataset · 2,400 rows")
)
themed_gt(formatted, n_rows=len(summary))

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: fmt_missing catches the NaN before the reader sees it</span><br><br>
Any column that can contain <code>NaN</code>: a score column with ~3% missing, an optional field: should have <code>fmt_missing(columns=..., missing_text=":")</code> added to the chain. A blank cell in a published table is ambiguous: did the student not sit the exam, or did the pipeline drop the value?
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Format the Program Table</span><br><br>
<b>Goal:</b> Take the <code>program_summary</code> from Activity 1 and add <code>cols_label</code>, <code>fmt_integer</code>, <code>fmt_number</code>, and <code>fmt_percent</code> to match the <code>formatted</code> table above.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>formatted_programs = (
    GT(program_summary)
    .cols_label(program="Program", n_students="Students", ...)
    .fmt_integer(columns="n_students")
    .fmt_number(columns=[...], decimals=1)
    .fmt_percent(columns="fail_rate", decimals=1)
)
themed_gt(formatted_programs, n_rows=len(program_summary))</pre>
</div>

In [ ]:
# TODO: add cols_label and fmt_* to your program_summary table
...

## 4. Column Spanners

When a table has several columns that belong to a natural group, for example three score columns or multiple model metrics, a column spanner adds a shared header label above the group. This reduces cognitive load: the reader understands the table structure before reading the individual values.

`tab_spanner(label, columns)` draws the label above the specified columns. It doesn't move or reorder columns; it only adds a visual grouping above them.


In [ ]:
# Course performance table: one row per course
course_detail = (
    df.groupby("course")
    .agg(
        students=("student_id", "count"),
        midterm=("midterm_score", "mean"),
        final=("final_score", "mean"),
        project=("project_score", "mean"),
        pass_rate=("passed", "mean"),
    )
    .reset_index()
    .round(2)
)
course_detail

In [ ]:
with_spanner = (
    GT(course_detail)
    .tab_header(title=gt_md("**Performance by Course**"))
    .cols_label(
        course="Course",
        students="Students",
        midterm="Midterm",
        final="Final",
        project="Project",
        pass_rate="Pass Rate",  # noqa: S106
    )
    .tab_spanner(label="Score (0-100)", columns=["midterm", "final", "project"])
    .fmt_integer(columns="students")
    .fmt_number(columns=["midterm", "final", "project"], decimals=1)
    .fmt_percent(columns="pass_rate", decimals=1)  # noqa: S106
    .tab_source_note("Source: DS-MLOps university analytics dataset · 2,400 rows")
)
themed_gt(with_spanner, n_rows=len(course_detail))

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Add a Spanner</span><br><br>
<b>Goal:</b> Take the <code>formatted_programs</code> table from Activity 2 and add a <code>tab_spanner</code> labelled <code>"Scores (0-100)"</code> over the three score columns.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>GT(program_summary)
    ...
    .tab_spanner(label="Scores (0-100)", columns=["midterm", "final", "project"])
    ...</pre>
</div>

In [ ]:
# TODO: add a tab_spanner to the program table
...

## 5. Conditional Styling

Conditional styling directs the reader's eye to the cells that matter: the highest pass rate, the lowest score, an outlier. `tab_style` applies a visual property and `loc` specifies exactly where it applies. `style` is the what, `loc` is the where.

The most common locations:
- `loc.body(columns, rows)`: specific cells in the data area
- `loc.column_labels()`: the column header row
- `loc.title()` / `loc.subtitle()`: the table header text

`rows` inside `loc.body()` accepts an integer index, a list of indices, or a lambda that receives the DataFrame and returns a boolean Series.


In [ ]:
highlighted = (
    GT(course_detail)
    .tab_header(title=gt_md("**Course Performance: Best and Worst Pass Rate**"))
    .cols_label(
        course="Course",
        students="Students",
        midterm="Midterm",
        final="Final",
        project="Project",
        pass_rate="Pass Rate",  # noqa: S106
    )
    .tab_spanner(label="Score (0-100)", columns=["midterm", "final", "project"])
    .fmt_integer(columns="students")
    .fmt_number(columns=["midterm", "final", "project"], decimals=1)
    .fmt_percent(columns="pass_rate", decimals=1)  # noqa: S106
    .tab_style(
        style=style.fill(color=SUCCESS),
        locations=loc.body(
            columns="pass_rate",
            rows=lambda df_gt: df_gt["pass_rate"] == df_gt["pass_rate"].max(),
        ),
    )
    .tab_style(
        style=style.fill(color="#FEF2F2"),
        locations=loc.body(
            columns="pass_rate",
            rows=lambda df_gt: df_gt["pass_rate"] == df_gt["pass_rate"].min(),
        ),
    )
)
themed_gt(highlighted, n_rows=len(course_detail))

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>loc</code> is a targeting system, not a filter</span><br><br>
<code>loc.body(rows=lambda df: df['pass_rate'] == df['pass_rate'].max())</code> doesn't subset the table: it identifies which rows receive the styling. The underlying data is unchanged. You can chain multiple <code>tab_style</code> calls; later ones add to earlier ones without overwriting.
</div>

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Passing a boolean mask directly to <code>rows</code></span><br><br>
<code>loc.body(rows=course_detail['pass_rate'] == course_detail['pass_rate'].max())</code> fails because <code>rows</code> inside <code>loc.body()</code> needs a callable that receives the <em>rendered</em> DataFrame, not the original one. Always use a lambda: <code>rows=lambda df: df['pass_rate'] == df['pass_rate'].max()</code>.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Highlight the Best Midterm Score</span><br><br>
<b>Goal:</b> Take the <code>highlighted</code> table and add a third <code>tab_style</code> call that highlights the <code>midterm</code> cell with the highest value in a light blue (<code>#EAF3FA</code>). Use a lambda for the row selection.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>.tab_style(
    style=style.fill(color="#EAF3FA"),
    locations=loc.body(
        columns="midterm",
        rows=lambda df: df["midterm"] == df["midterm"].max(),
    ),
)</pre>
</div>

In [ ]:
# TODO: add a third tab_style call for the highest midterm value
...

## 6. Summary Rows

A summary row aggregates the entire table into one footer row: a grand mean, a column total, or a count. The reader no longer needs to mentally compute the aggregate, and the table and its summary stay in the same visual unit.

`grand_summary_rows(fns)` adds these rows. `fns` is a dict mapping a display label to an aggregation function. In version 0.20, it aggregates all numeric columns in the table, so the DataFrame passed to `GT` should contain only the columns you want summarised:


In [ ]:
from great_tables import vals as gt_vals  # noqa: F401

# Use only the score + pass_rate columns so the summary row is meaningful
course_scores = course_detail.drop(columns=["students"])

with_summary = (
    GT(course_scores)
    .tab_header(title=gt_md("**Course Summary with Grand Mean**"))
    .cols_label(
        course="Course",
        midterm="Midterm",
        final="Final",
        project="Project",
        pass_rate="Pass Rate",  # noqa: S106
    )
    .tab_spanner(label="Score (0-100)", columns=["midterm", "final", "project"])
    .fmt_number(columns=["midterm", "final", "project"], decimals=1)
    .fmt_percent(columns="pass_rate", decimals=1)  # noqa: S106
    .grand_summary_rows(
        fns={"Mean": lambda x: x.mean(numeric_only=True)},
    )
)
themed_gt(with_summary, n_rows=len(course_scores))

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Shape the DataFrame before passing it to GT</span><br><br>
<code>grand_summary_rows</code> aggregates every numeric column in the table. If a count column like <code>students</code> would produce a meaningless mean, drop it before calling <code>GT()</code>: <code>df.drop(columns=["students"])</code>. If the table still includes a string column like the row label, pass <code>numeric_only=True</code> to the aggregation: <code>lambda x: x.mean(numeric_only=True)</code>.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5 - Add a Min and Max Row</span><br><br>
<b>Goal:</b> Extend <code>with_summary</code> to show three summary rows: <code>Min</code>, <code>Max</code>, and <code>Mean</code> across all score columns. Pass a dict with three keys to <code>fns</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>fns={"Min": lambda x: x.min(numeric_only=True), "Max": lambda x: x.max(numeric_only=True), "Mean": lambda x: x.mean(numeric_only=True)}</pre>
</div>

In [ ]:
# TODO: add Min/Max/Mean grand summary rows
...

## 7. Inline Sparklines with `fmt_nanoplot`

A table that shows a pass rate of 72.4% answers "how many passed?" It doesn't show whether
that rate improved over five semesters or collapsed. A sparkline -- a mini line chart inside
a single cell -- answers both questions at once without opening a separate figure.

`fmt_nanoplot` transforms a column of Python lists into inline micro-charts. The pattern is
two steps:

1. Add a list-valued column: each cell holds a sequence of numbers (one per period, per
   category, or per experiment)
2. Chain `.fmt_nanoplot(columns='col_name', plot_type='line')` into your GT chain

`plot_type` accepts `'line'` (trends over time) or `'bar'` (magnitude across categories).
`nanoplot_options` controls colours, point visibility, and hover labels.

In [ ]:
rng: np.random.Generator = np.random.default_rng(42)
N_SEMESTERS: int = 5

# Simulate a rising score trajectory with noise for each course
trajectories: list[list[float]] = []
for _, row in course_detail.iterrows():
    start: float = row["midterm"] - rng.uniform(4, 8)
    end: float = row["midterm"] + rng.uniform(2, 10)
    noise: np.ndarray = rng.normal(0, 1.5, N_SEMESTERS)
    traj: list[float] = [round(float(v), 1) for v in np.linspace(start, end, N_SEMESTERS) + noise]
    trajectories.append(traj)

course_trend: pd.DataFrame = course_detail[["course", "midterm", "pass_rate"]].copy()
course_trend["score_trend"] = trajectories
course_trend.head(3)

# fmt_nanoplot requires a Polars DataFrame for list columns
# (pd.isna on Python lists in pandas 3.x returns an array, not a scalar)
course_trend_pl: pl.DataFrame = pl.from_pandas(course_trend)
course_trend_pl.head(3)

In [ ]:
spark_table: GT = (
    GT(course_trend_pl)
    .tab_header(
        title=gt_md("**Midterm Score Trend by Course**"),
        subtitle="Sparkline shows the estimated 5-semester score trajectory",
    )
    .cols_label(
        course="Course",
        midterm="Mean Midterm",
        pass_rate="Pass Rate",  # noqa: S106
        score_trend="Score Trend",
    )
    .fmt_number(columns="midterm", decimals=1)
    .fmt_percent(columns="pass_rate", decimals=1)  # noqa: S106
    .fmt_nanoplot(
        columns="score_trend",
        plot_type="line",
        options=nanoplot_options(
            data_point_fill_color=PRIMARY,
            data_line_stroke_color=PRIMARY,
            show_data_area=False,
            interactive_data_values=True,
        ),
    )
    .tab_source_note("Trajectories are simulated; real data would use per-semester aggregations")
)
themed_gt(spark_table, n_rows=len(course_trend))

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>fmt_nanoplot</code> requires a list-valued column</span><br><br>
<code>fmt_nanoplot</code> doesn't aggregate data for you. Pass a <strong>Polars DataFrame</strong> to <code>GT()</code> when your column holds Python lists: Polars types them as <code>List(Float64)</code>, which GT renders without ambiguity. Pandas 3.x <code>pd.isna(list)</code> returns an array that causes a <code>ValueError</code> inside the nanoplot renderer. If your data is in wide format (one column per semester),
convert it first with <code>.values.tolist()</code> row-wise or with a list comprehension,
then assign the result as a new column before calling <code>GT()</code>.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 7 - Switch to a Bar Sparkline</span><br><br>
<b>Goal:</b> Copy <code>spark_table</code> and change two things: set <code>plot_type='bar'</code>
and set <code>show_data_area=True</code> inside <code>nanoplot_options</code>. Compare the two
chart types side by side. Which makes it easier to spot the course with the largest single-semester
score peak? Which better communicates overall trend direction?<br><br>
<b>Hint:</b> Bar nanoplots colour negative bars differently by default. Try setting
<code>show_data_points=False</code> to clean up the bar variant.
</div>

In [ ]:
# TODO: build a bar nanoplot version of spark_table
...

## 8. Model Comparison with `metrics_report()`

The `ark.plot.gt_style` module ships `metrics_report()`: a one-call wrapper that produces a publication-ready model comparison table. It handles formatting, brand styling, and conditional highlighting in a single call.

`metrics_report(df, metrics, minimize_cols, maximize_cols)` highlights the best value in each metric column: green for minimize metrics (lower is better: MAE, RMSE), green for maximize metrics (higher is better: R², accuracy). The caller decides which direction is better for each metric; the function doesn't guess.


In [ ]:
comparison = pd.DataFrame(
    {
        "Model": ["Linear Regression", "Ridge (α=0.1)", "Ridge (α=1.0)", "Random Forest"],
        "MAE": [8.21, 8.09, 7.98, 7.43],
        "RMSE": [10.42, 10.31, 10.19, 9.61],
        "R2": [0.781, 0.784, 0.788, 0.810],
    }
)

metrics_report(
    comparison,
    metrics=["MAE", "RMSE", "R2"],
    minimize_cols=["MAE", "RMSE"],
    maximize_cols=["R2"],
    title="Grade Prediction: Model Comparison",
    subtitle="Predicting average_marks from study_hours, attendance_pct, and program",
    source_note="university_analytics.csv · 5-fold CV · held-out 20% test set",
)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: metrics_report highlights by direction, not by rank</span><br><br>
<code>minimize_cols</code> highlights the row with the <em>lowest</em> value: better for error metrics. <code>maximize_cols</code> highlights the row with the <em>highest</em> value: better for performance metrics. A column can appear in at most one list. If a column appears in neither, it's formatted but not highlighted.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 6 - Add a Gradient Boosting Row</span><br><br>
<b>Goal:</b> Add a fifth row to <code>comparison</code>: <code>"Gradient Boosting"</code> with MAE=6.91, RMSE=8.84, R2=0.843: and re-run <code>metrics_report()</code>. Confirm the highlighted row updates automatically.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>comparison = pd.concat([comparison, pd.DataFrame([{"Model": "Gradient Boosting", "MAE": 6.91, "RMSE": 8.84, "R2": 0.843}])], ignore_index=True)
metrics_report(comparison, metrics=[...], minimize_cols=[...], maximize_cols=[...], ...)</pre>
</div>

In [ ]:
# TODO: add Gradient Boosting row and re-run metrics_report
...

## Capstone: Course Performance Report

Combine every technique from this notebook into one complete report table. The report should give a department head a single table they can paste into a slide deck.


<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Course Performance Report</span><br><br>

<b>Goal:</b>
<ol>
<li>Build a <code>report</code> DataFrame grouped by <code>course</code> with columns: students, midterm mean, final mean, project mean, pass rate, and average_marks mean</li>
<li>Wrap with <code>GT</code>. Add a descriptive title and source note</li>
<li>Apply <code>cols_label</code> and the appropriate <code>fmt_*</code> for each column</li>
<li>Add a <code>tab_spanner</code> over the three score columns</li>
<li>Highlight the course with the highest pass rate (green) and lowest pass rate (light red)</li>
<li>Add a grand <code>Mean</code> summary row across all numeric columns</li>
<li>Call <code>themed_gt()</code> last</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'># Build report DataFrame first, then chain all GT methods in one expression</pre>
</div>

In [ ]:
# TODO: build the complete course performance report
...

## Further Reading

| Resource | Why it matters |
|---|---|
| [Great Tables: `fmt_nanoplot` reference](https://posit-dev.github.io/great-tables/reference/GT.fmt_nanoplot.html) | Parameters and options for inline sparklines; includes line, bar, and boxplot types |
| [Great Tables documentation](https://posit-dev.github.io/great-tables/) | Complete API reference with rendered examples for every method |
| [Great Tables: `loc` reference](https://posit-dev.github.io/great-tables/reference/#targeting-cells) | Full list of location helpers: `loc.body`, `loc.column_labels`, `loc.spanner_labels`, `loc.grand_summary` |
| [Great Tables blog: Python tables](https://posit-dev.github.io/great-tables/blog/) | Worked examples including financial reports and ML comparison tables |
| Knaflic, C.N. (2015). *Storytelling with Data*. Wiley. | Chapter 2 covers when tables serve communication better than charts |
| [pandas `GroupBy.agg` reference](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html) | Named aggregations (`col=(src, func)`) used throughout this notebook |


## Summary

| GT method | What it does |
|---|---|
| `GT(df)` | Wrap a DataFrame and begin the method chain |
| `themed_gt(table, n_rows=n)` | Apply project brand: header colors, font, striped rows. Call last. |
| `tab_header(title, subtitle)` | Add a title row above the column headers |
| `tab_source_note(text)` | Add an attribution line below the table |
| `cols_label(**kwargs)` | Replace column names with reader-facing labels |
| `fmt_number(columns, decimals)` | Round floats to `decimals` places |
| `fmt_integer(columns)` | Remove decimal, add thousands separator |
| `fmt_percent(columns, decimals)` | Multiply by 100 and append `%` |
| `fmt_missing(columns, missing_text)` | Replace `NaN` with a readable placeholder |
| `tab_spanner(label, columns)` | Group related columns under a shared header label |
| `tab_style(style, locations)` | Apply a visual property (`fill`, `text`) to a location (`loc.body`, `loc.column_labels`) |
| `loc.body(columns, rows)` | Target specific cells; `rows` takes an index or a lambda |
| `grand_summary_rows(fns)` | Add one summary row per key in `fns`; aggregates all numeric columns |
| `metrics_report(df, metrics, ...)` | One-call ML comparison table with directional highlighting |

**Next:** Part 3: Dev Tools covers the professional toolchain: uv, ruff, type annotations, git, pytest, and pre-commit.
